# Unemployment Analysis with Python

## tl;dr
Analyze the uploaded monthly Indian unemployment data, with emphasis on cleaning, Covid-19 disruption, urban/rural differences, and seasonality.

## Context & Methods
- Pre-Covid: before April 2020.
- Covid shock window: April-May 2020.
- June 2020 is treated as a recovery/normalization month.
- Averages are equal-weight means across available state-area observations, not official population-weighted national rates.


In [1]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

source_path = Path("/mnt/data/Pasted markdown.md")
text = source_path.read_text(errors="replace")

pattern = re.compile(
    r'(?P<state>[A-Za-z& ]+?)(?P<date>\d{2}-\d{2}-\d{4})Monthly'
    r'(?P<unemp>\d{1,2}\.\d{2})(?P<employed>\d+\.00)'
    r'(?P<lfpr>\d{1,2}\.\d{2})(?P<area>Urban|Rural)'
)
rows = [m.groupdict() for m in pattern.finditer(text)]
df = pd.DataFrame(rows)
df["Date"] = pd.to_datetime(df["date"], dayfirst=True)
df["Unemployment_Rate"] = pd.to_numeric(df["unemp"])
df["Estimated_Employed"] = pd.to_numeric(df["employed"])
df["Labour_Participation_Rate"] = pd.to_numeric(df["lfpr"])
df["State"] = df["state"]
df["Area"] = df["area"]
df = df[["State","Date","Unemployment_Rate","Estimated_Employed","Labour_Participation_Rate","Area"]]
df = df.sort_values(["Date","State","Area"]).reset_index(drop=True)
df.head()


,State,Date,Unemployment_Rate,Estimated_Employed,Labour_Participation_Rate,Area
0,Andhra Pradesh,2019-05-31,3.65,11999139.0,43.24,Rural
1,Andhra Pradesh,2019-05-31,6.09,4788661.0,37.45,Urban
2,Assam,2019-05-31,4.29,11749334.0,57.39,Rural
3,Assam,2019-05-31,7.87,1671707.0,44.92,Urban
4,Bihar,2019-05-31,9.27,24322330.0,39.75,Rural


## Data Quality

In [2]:
print("Rows:", len(df))
print("States/UTs:", df["State"].nunique())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Duplicate state-area-date keys:", df.duplicated(["State","Area","Date"]).sum())
print("Unemployment range:", df["Unemployment_Rate"].min(), "to", df["Unemployment_Rate"].max())

expected_months = pd.date_range(df["Date"].min(), df["Date"].max(), freq="ME")
coverage = []
for (state, area), g in df.groupby(["State","Area"]):
    coverage.append({
        "State": state,
        "Area": area,
        "Observed_Months": g["Date"].nunique(),
        "Missing_Months": len(set(expected_months) - set(g["Date"]))
    })
pd.DataFrame(coverage).sort_values("Missing_Months", ascending=False).head(10)


Rows: 740
States/UTs: 28
Date range: 2019-05-31 to 2020-06-30
Duplicate state-area-date keys: 0
Unemployment range: 0.0 to 76.74


,State,Area,Observed_Months,Missing_Months
41,Sikkim,Rural,5,9
20,Jammu & Kashmir,Urban,10,4
19,Jammu & Kashmir,Rural,11,3
11,Goa,Rural,12,2
12,Goa,Urban,12,2
6,Chandigarh,Urban,12,2
2,Assam,Rural,12,2
35,Puducherry,Rural,12,2
42,Sikkim,Urban,12,2
32,Meghalaya,Urban,13,1


## Results

In [3]:
monthly = df.groupby("Date")["Unemployment_Rate"].agg(
    Mean="mean", Median="median", Observations="size"
).reset_index()
monthly


,Date,Mean,Median,Observations
0,2019-05-31,8.874259,6.870,54
1,2019-06-30,9.303333,7.570,54
2,2019-07-31,9.033889,7.160,54
3,2019-08-31,9.637925,7.270,53
4,2019-09-30,9.051731,6.240,52
5,2019-10-31,9.900909,7.290,55
6,2019-11-30,9.868364,6.940,55
7,2019-12-31,9.497358,7.240,53
8,2020-01-31,9.950755,6.790,53
9,2020-02-29,9.964717,7.550,53


In [4]:
pre = df[df["Date"] < "2020-04-01"]["Unemployment_Rate"].mean()
apr_may = df[df["Date"].between("2020-04-01","2020-05-31")]["Unemployment_Rate"].mean()
june = df[df["Date"] == "2020-06-30"]["Unemployment_Rate"].mean()

pd.DataFrame({
    "Period": ["Pre-Covid", "Apr-May 2020", "Jun 2020"],
    "Mean_Unemployment_Rate": [pre, apr_may, june],
    "Change_vs_Pre_Covid_pp": [0, apr_may-pre, june-pre]
}).round(2)


,Period,Mean_Unemployment_Rate,Change_vs_Pre_Covid_pp
0,Pre-Covid,9.61,0.00
1,Apr-May 2020,24.26,14.64
2,Jun 2020,11.90,2.29


In [5]:
area = (
    df.assign(Period=np.select(
        [df["Date"] < "2020-04-01",
         df["Date"].between("2020-04-01","2020-05-31"),
         df["Date"] >= "2020-06-01"],
        ["Pre-Covid","Apr-May 2020","Jun 2020"], default="Other"))
    .groupby(["Area","Period"])["Unemployment_Rate"].mean().unstack()
)
area.round(2)


Period,Apr-May 2020,Jun 2020,Pre-Covid
Area,,,
Rural,21.48,11.83,8.23
Urban,26.93,11.98,10.91


In [6]:
baseline = df[df["Date"] < "2020-04-01"].groupby(["State","Area"])["Unemployment_Rate"].mean()
shock = df[df["Date"].between("2020-04-01","2020-05-31")].groupby(["State","Area"])["Unemployment_Rate"].mean()
change = pd.concat([baseline.rename("Pre_Covid"), shock.rename("Apr_May_2020")], axis=1).dropna()
change["Change_pp"] = change["Apr_May_2020"] - change["Pre_Covid"]
change.sort_values("Change_pp", ascending=False).head(10).round(2)


Pre_Covid  Apr_May_2020  Change_pp
State          Area                                     
Puducherry     Urban       2.01         75.87      73.86
               Rural       1.15         74.51      73.36
Jharkhand      Urban      19.30         65.82      46.53
Tamil Nadu     Rural       2.63         45.96      43.33
Jharkhand      Rural       8.61         48.41      39.80
Bihar          Rural      11.09         46.18      35.09
Tamil Nadu     Urban       3.70         35.75      32.05
Bihar          Urban      16.89         48.32      31.43
Karnataka      Rural       3.03         28.44      25.42
Andhra Pradesh Urban       6.27         28.60      22.34

## Seasonality

Only about one partial annual cycle is available, so seasonality should be treated as exploratory. Use additional years of data for formal seasonal decomposition.


In [7]:
seasonality = (
    df[df["Date"] < "2020-04-01"]
    .assign(Month=df["Date"].dt.month)
    .groupby("Month")["Unemployment_Rate"]
    .agg(["mean","median","count"])
)
seasonality.round(2)


,mean,median,count
Month,,,
1,9.95,6.79,53
2,9.96,7.55,53
3,10.70,8.53,52
5,8.87,6.87,54
6,9.30,7.57,54
7,9.03,7.16,54
8,9.64,7.27,53
9,9.05,6.24,52
10,9.90,7.29,55


## Takeaways

- April-May 2020 is the clearest break in the observed series.
- The equal-weight mean rises from about 9.6% pre-Covid to about 24.3% in April-May, then falls to about 11.9% in June.
- Urban and rural observations both experience large increases.
- The Covid shock varies substantially by state-area group.
- The available history is not sufficient to establish robust annual seasonality.
